# Лабораторная 03. Ответы по регламентам DWH
Сколько дней пересчитываются возвраты и какое расхождение выручки блокирует публикацию? Ответы определяет локальный справочник вымышленной сети «Маяк». Одна модель кодирует разделы Markdown и вопросы, другая читает до двух найденных фрагментов и отвечает. Под ответом чат показывает имена файлов и разделов. [Инструкция по запуску](../README.md).

In [1]:
import atexit
from contextlib import ExitStack
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import display
from dwh_agent.fact_agent import FactAgent
from dwh_agent.fact_memory import FileFactMemory
from dwh_agent.local_server import LocalServer
from dwh_agent.notebook_ui import NotebookChat
from dwh_agent.paths import LAB3_DIR, VAR_DIR
from dwh_agent.rag import QwenEncoder, VectorIndex

## Запуск и индексация
При первом запуске энкодер обрабатывает документы из `knowledge/`. Файл SQLite в `var/` можно использовать в следующих сеансах. После правки документов повторите эту ячейку. Параметр `CPU = True` включает режим CPU для обеих моделей.

In [2]:
CPU = False
if "servers" in globals():
    servers.close()
servers = ExitStack()
try:
    for backend in ("local-embedding", "local-4b"):
        servers.enter_context(LocalServer(backend, cpu=CPU))
    index = VectorIndex(VAR_DIR / "rag.sqlite3", QwenEncoder())
    count = index.sync(LAB3_DIR / "knowledge")
    print(f"В индексе {count} разделов. Энкодер: {index.encoder.model}.")
except BaseException:
    servers.close()
    raise
atexit.register(servers.close);

В индексе 6 разделов. Энкодер: qwen3-embedding-0.6b.


## Проверка цикла вопросов и ответов
Последовательно задаём вопрос по регламенту, уточняем исключение, меняем тему и просим сведения вне базы. Вывод содержит реальные ответы и косинусную близость найденных разделов. Профиль примера хранится во временном файле.

In [3]:
questions = [
    ("За сколько дней ночной процесс повторно обрабатывает возвраты?", "sales.md", "35"),
    ("А если покупку вернули позже этого срока?", "sales.md", "чек"),
    ("При какой разнице между кассой и витриной блокируется публикация?", "operations.md", "7"),
    ("Какова температура на поверхности Венеры?", None, "баз"),
]
with TemporaryDirectory() as directory:
    events = []
    demo = FactAgent(FileFactMemory(Path(directory) / "facts.json"), retriever=index, on_event=events.append)
    for question, expected_file, expected_term in questions:
        result = demo.ask(question)
        retrieval = next(e for e in reversed(events) if e["event"] == "rag_retrieved")
        print("Вопрос:", question)
        print("Ответ:", result["answer"])
        print("Разделы и оценки:", list(zip(result["sources"], retrieval["scores"])), "\n")
        if expected_file:
            assert result["sources"] and result["sources"][0].startswith(expected_file)
        else:
            assert result["sources"] == []
        assert expected_term in result["answer"].casefold(), "Проверьте смысл ответа и выбранный раздел"

Вопрос: За сколько дней ночной процесс повторно обрабатывает возвраты?
Ответ: Ночной процесс повторно обрабатывает возвраты за последние 35 календарных дней.
Разделы и оценки: [('sales.md — Поздние возвраты и горизонт пересчёта', 0.716)] 



Вопрос: А если покупку вернули позже этого срока?
Ответ: Если возврат происходит позже срока в 35 календарных дней, оператор запускает адресный пересчёт по номеру чека. Ночной процесс в этом случае не обрабатывает возврат автоматически.
Разделы и оценки: [('sales.md — Поздние возвраты и горизонт пересчёта', 0.727)] 



Вопрос: При какой разнице между кассой и витриной блокируется публикация?
Ответ: Публикация блокируется при абсолютном расхождении более 7 рублей. При расхождении ровно 7 рублей публикация разрешена.
Разделы и оценки: [('operations.md — Карантин при сверке выручки', 0.693)] 



Вопрос: Какова температура на поверхности Венеры?
Ответ: В базе знаний нет сведений по этому вопросу.
Разделы и оценки: [] 



## Свой диалог
Например, спросите «Как выбрать версию поставщика на момент продажи?». В служебной ячейке отображаются результаты поиска и события памяти. Команда `/exit` закрывает оба локальных сервера.

In [4]:
ui = NotebookChat()
display(ui.service_output)

In [5]:
chat = FactAgent(FileFactMemory(VAR_DIR / "rag_memory.json", "student"), retriever=index, on_event=ui.service)
ui.attach(chat, on_close=servers.close)
display(ui.chat_box)

In [6]:
if os.environ.get("DWH_NOTEBOOK_VERIFY") == "1":
    servers.close()